# 01 · Data Loading & Cleaning
**Climate Trend Analyzer** — step 1 of 5

### Why this step matters
Real climate data is *never* clean. A dataset that arrives from a station
network, a CSV download or a sensor gateway will contain:

* missing readings (sensor dropouts, failed API calls)
* duplicate records (retried uploads)
* impossible values (a `-999 °C` placeholder, a unit error)
* inconsistent date formats (`2020-01-01` **and** `01/31/2020`)

If we skip cleaning, every chart is misleading and every model learns from
garbage. In this notebook we load the dataset, **measure** the problems,
**fix** them, and export a clean master table.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Launch Jupyter from the project root or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

from src import config as cfg
from src import data_loader, preprocessing, features, eda, trend_analysis
from src import anomaly_detection, forecasting, insights, visualizations as viz

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)

Project root: /home/user/Climate-Trend-Analyzer


## 1.1 Load the raw dataset

In [2]:
raw = data_loader.load_raw(cfg.RAW_CSV)
print(data_loader.dataset_info(raw))

[data_loader] Loaded raw dataset: /home/user/Climate-Trend-Analyzer/data/raw/climate_raw.csv -> shape (2715, 14)
DATASET PROFILE  shape=(2715, 14)
date             object
region           object
country          object
latitude        float64
longitude       float64
temp_mean_c     float64
temp_min_c      float64
temp_max_c      float64
rainfall_mm     float64
humidity_pct    float64
co2_ppm         float64
sea_level_mm    float64
sim_heatwave      int64
sim_drought       int64
--------------------------------------------------------------
Missing values:
date             0
region           0
country          0
latitude         0
longitude        0
temp_mean_c     40
temp_min_c       0
temp_max_c       0
rainfall_mm     40
humidity_pct    40
co2_ppm          0
sea_level_mm     0
sim_heatwave     0
sim_drought      0
--------------------------------------------------------------
Duplicate rows: 13


> **Look closely at the date column** - it is a *string* column containing
> two different formats. That is the first problem we have to solve.

## 1.2 Measure the damage

In [3]:
missing = raw[cfg.CLIMATE_COLUMNS].isna().sum().to_frame("missing")
missing["missing_pct"] = (100 * missing["missing"] / len(raw)).round(2)
missing

,missing,missing_pct
temp_mean_c,40,1.47
temp_min_c,0,0.00
temp_max_c,0,0.00
rainfall_mm,40,1.47
humidity_pct,40,1.47
co2_ppm,0,0.00
sea_level_mm,0,0.00


In [4]:
# how many rows are physically impossible?
for col, (lo, hi) in cfg.VALID_RANGES.items():
    bad = raw[col].notna() & ((raw[col] < lo) | (raw[col] > hi))
    print(f"{col:15s} -> {int(bad.sum()):4d} impossible values")

temp_mean_c     ->   12 impossible values
temp_min_c      ->    0 impossible values
temp_max_c      ->    0 impossible values
rainfall_mm     ->    0 impossible values
humidity_pct    ->    0 impossible values
co2_ppm         ->    0 impossible values
sea_level_mm    ->    0 impossible values


## 1.3 Clean it

In [5]:
clean, report = preprocessing.clean_dataset(raw)
print(preprocessing.quality_report_table(report).to_string(index=False))

                               metric  value
                    Rows loaded (raw) 2715.0
Rows dropped (bad dates / duplicates)   15.0
                Rows in clean dataset 2700.0
            Unparseable dates removed    0.0
               Duplicate rows removed   15.0
    Rows containing impossible values   12.0
                 Total values imputed  132.0
      Completeness after cleaning (%)  100.0


In [6]:
# the dates are real timestamps now, and the outliers are gone
clean["date"].head(), clean["temp_mean_c"].describe()

(0   1980-01-01
 1   1980-02-01
 2   1980-03-01
 3   1980-04-01
 4   1980-05-01
 Name: date, dtype: datetime64[ns],
 count    2700.000000
 mean       20.151848
 std         8.775462
 min        -1.100000
 25%        13.797500
 50%        21.950000
 75%        27.410000
 max        39.000000
 Name: temp_mean_c, dtype: float64)

## 1.4 Visual proof of the cleaning

In [7]:
viz.plot_data_quality(raw, clean)

[viz] saved 01_data_quality.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/01_data_quality.png')

### ✅ Checkpoint
* 2,700 clean monthly records, one row per (region, month)
* 0 missing values, 0 impossible values, 0 duplicate dates
* `date` is a proper `datetime64` column

In [8]:
clean.to_csv(cfg.CLEAN_CSV, index=False)
print("Saved ->", cfg.CLEAN_CSV)

Saved -> /home/user/Climate-Trend-Analyzer/data/processed/climate_clean.csv
